In [1]:
"""
Count SPVM criminal acts (Actes criminels) by crime category in each of the
19 Montreal boroughs, 2023-2025.

  - Actes criminels CSV        https://donnees.montreal.ca/dataset/actes-criminels
  - Borough boundaries GeoJSON https://donnees.montreal.ca/dataset/limites-administratives-agglomeration

Inputs (downloaded by scripts/download_sources.py):
  - data/shirley/raw/actes-criminels.csv
  - data/sebastien/raw/bourough-geo-coords.json

Outputs (in data/shirley/processed/):
  - crimes_by_borough_2023_2025.csv      borough x category, 2023-2025 pooled
  - crimes_by_borough_by_year.csv        borough x year x category (long format)
  - crimes_join_report.txt               how many records were kept / lost and why, plus retrieval date for each source
"""

import json
from pathlib import Path
from datetime import datetime

import pandas as pd
import geopandas as gpd

# Works whether the notebook runs from the repo root or from scripts/.
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "data").is_dir())
DATA = ROOT / "data"
OUT_DIR = DATA / "shirley" / "processed"
MANIFEST = DATA / "sources_manifest.json"

CRIME_CSV = DATA / "shirley" / "raw" / "actes-criminels.csv"
CRIME_URL = (
    "https://donnees.montreal.ca/dataset/5829b5b0-ea6f-476f-be94-bc2b8797769a/"
    "resource/c6f482bf-bf0f-4960-8b2f-9982c211addd/download/actes-criminels.csv"
)
BOROUGH_FILE = DATA / "sebastien" / "raw" / "bourough-geo-coords.json"
BOROUGH_URL = (
    "https://donnees.montreal.ca/dataset/9797a946-9da8-41ec-8815-f6b276dec7e9/"
    "resource/e18bfd07-edc8-4ce8-8a5a-3b617662a794/download/"
    "limites-administratives-agglomeration.geojson"
)

YEARS = [2023, 2024, 2025]

# English labels for the six categories in the source
CATEGORY_EN = {
    "Introduction": "Break-in",
    "Vol dans / sur véhicule à moteur": "Theft from vehicle",
    "Vol de véhicule à moteur": "Vehicle theft",
    "Méfait": "Mischief",
    "Vols qualifiés": "Robbery",
    "Infractions entrainant la mort": "Offence causing death",
}

# Bounding box of the island of Montreal (WGS84 coordinate system).
# Points outside are treated as having no usable location
# the source uses placeholder values for events without a location.
LAT_MIN, LAT_MAX = 45.35, 45.75
LON_MIN, LON_MAX = -74.05, -73.40


def read_csv_any_encoding(path: Path) -> pd.DataFrame:
    for enc in ("utf-8", "utf-8-sig", "latin-1"):
        try:
            return pd.read_csv(path, encoding=enc, low_memory=False)
        except UnicodeDecodeError:
            continue
    raise RuntimeError(f"Could not decode {path}")


def retrieval_dates() -> dict:
    """File name -> retrieval date, from the manifest written by download_sources.py."""
    for path in (CRIME_CSV, BOROUGH_FILE):
        if not path.exists():
            raise SystemExit(f"{path.relative_to(ROOT)} not found: run scripts/download_sources.py first")
    sources = json.loads(MANIFEST.read_text(encoding="utf-8"))["sources"]
    return {Path(s["file"]).name: s["retrieved"] for s in sources}


def load_boroughs() -> gpd.GeoDataFrame:
    gdf = gpd.read_file(BOROUGH_FILE).to_crs(epsg=4326)      # code for WGS 84
    is_borough = gdf["TYPE"].str.lower().str.startswith("arrondissement")
    boroughs = gdf.loc[is_borough, ["NOM", "geometry"]].rename(columns={"NOM": "borough"})
    others = gdf.loc[~is_borough, ["NOM", "TYPE", "geometry"]]

    n = boroughs["borough"].nunique()
    if n != 19:
        print(f"Expected 19 boroughs, found {n}:")
        print(sorted(boroughs["borough"].unique()))
        print(f"TYPE values in file: {gdf['TYPE'].unique().tolist()}")
    return boroughs, others


def main():
    retrieved = retrieval_dates()
    OUT_DIR.mkdir(parents=True, exist_ok=True)

    report = []
    log = lambda s="": (print(s), report.append(s))

    log(f"Run date: {datetime.now():%Y-%m-%d %H:%M}")
    log("Sources (licence: CC BY 4.0, Ville de Montréal):")
    log(f"   {CRIME_CSV.name}  retrieved {retrieved[CRIME_CSV.name]}  <- {CRIME_URL}")
    log(f"   {BOROUGH_FILE.name}  retrieved {retrieved[BOROUGH_FILE.name]}  <- {BOROUGH_URL}")
    log("")

    # Load and filter to 2023-2025
    df = read_csv_any_encoding(CRIME_CSV)
    df.columns = [c.strip().upper() for c in df.columns]
    log(f"Rows in source file (all years): {len(df):,}")

    # check dates
    df["DATE"] = pd.to_datetime(df["DATE"], errors="coerce")
    bad_dates = df["DATE"].isna().sum()
    if bad_dates:
        log(f"Rows with unparseable DATE (dropped): {bad_dates:,}")
    # add year column
    df = df[df["DATE"].dt.year.isin(YEARS)].copy()
    df["YEAR"] = df["DATE"].dt.year
    n_window = len(df)
    log(f"Rows in {YEARS[0]}-{YEARS[-1]}: {n_window:,}")
    # count rows for each year
    for y in YEARS:
        log(f"   {y}: {(df['YEAR'] == y).sum():,}")
    log("")

    # Drop records without a usable location
    df["LATITUDE"] = pd.to_numeric(df["LATITUDE"], errors="coerce")
    df["LONGITUDE"] = pd.to_numeric(df["LONGITUDE"], errors="coerce")
    has_loc = (
        df["LATITUDE"].between(LAT_MIN, LAT_MAX)
        & df["LONGITUDE"].between(LON_MIN, LON_MAX)
    )
    no_loc = df[~has_loc]
    log(f"No usable location (missing or placeholder coordinates): {len(no_loc):,} "
        f"({len(no_loc) / n_window:.1%})")
    if len(no_loc):
        log("   by category:")
        for cat, k in no_loc["CATEGORIE"].value_counts().items():
            log(f"      {cat}: {k:,}")
    df = df[has_loc]

    pts = gpd.GeoDataFrame(
        df, geometry=gpd.points_from_xy(df["LONGITUDE"], df["LATITUDE"]), crs="EPSG:4326"
    )
    pts["_rid"] = range(len(pts))

    # Spatial join to boroughs
    boroughs, others = load_boroughs()

    joined = gpd.sjoin(pts, boroughs, how="left", predicate="intersects")

    # Points are snapped to street intersections
    # so some sit exactly on a borough boundary and match two polygons.
    # Keep the first match only.
    dup = joined["_rid"].duplicated(keep="first")
    n_boundary = joined.loc[dup, "_rid"].nunique()
    joined = joined[~dup]
    log("")
    log(f"Points on a borough boundary (matched 2 boroughs, kept first): {n_boundary:,}")

    unmatched = joined[joined["borough"].isna()]
    matched = joined[joined["borough"].notna()]

    # Why didn't the unmatched ones fall in a borough?
    um = gpd.sjoin(
        unmatched.drop(columns=["index_right", "borough"]),
        others.rename(columns={"NOM": "other_name"}),
        how="left", predicate="intersects",
    ).drop_duplicates("_rid")
    in_linked_city = um["other_name"].notna()
    log(f"Located but outside the 19 boroughs: {len(um):,}")
    log(f"   in a linked city (ville liée), e.g. Westmount, Mont-Royal: {in_linked_city.sum():,}")
    for name, k in um.loc[in_linked_city, "other_name"].value_counts().items():
        log(f"      {name}: {k:,}")
    log(f"   outside all agglomeration polygons (water, edge cases): {(~in_linked_city).sum():,}")
    log("")

    if "PDQ" in matched.columns:
        metro = (pd.to_numeric(matched["PDQ"], errors="coerce") == 50).sum()
        log(f"Kept records attributed to PDQ 50 (metro, placed at station entrances): {metro:,}")

    log(f"KEPT in the 19 boroughs: {len(matched):,} of {n_window:,} "
        f"({len(matched) / n_window:.1%})")
    log("")

    # Tables
    matched = matched.assign(category=matched["CATEGORIE"].map(CATEGORY_EN)
                             .fillna(matched["CATEGORIE"]))

    unknown = set(matched["CATEGORIE"].unique()) - set(CATEGORY_EN)
    if unknown:
        log(f"NOTE: categories not in the English map (kept as-is): {sorted(unknown)}")

    all_boroughs = sorted(boroughs["borough"].unique())

    pooled = (
        matched.pivot_table(index="borough", columns="category",
                            values="_rid", aggfunc="count", fill_value=0)
        .reindex(all_boroughs, fill_value=0)
    )
    pooled["Total"] = pooled.sum(axis=1)
    pooled = pooled.sort_index()
    pooled.to_csv(OUT_DIR / "crimes_by_borough_2023_2025.csv", encoding="utf-8-sig")

    by_year = (
        matched.groupby(["borough", "YEAR", "category"]).size()
        .rename("count").reset_index()
        .sort_values(["borough", "YEAR", "category"])
    )
    by_year.to_csv(OUT_DIR / "crimes_by_borough_by_year.csv", index=False, encoding="utf-8-sig")

    (OUT_DIR / "crimes_join_report.txt").write_text("\n".join(report), encoding="utf-8")

    pd.set_option("display.width", 200)
    pd.set_option("display.max_columns", 20)
    print(pooled)
    print(f"\nWrote crimes_by_borough_2023_2025.csv, crimes_by_borough_by_year.csv, "
          f"crimes_join_report.txt to {OUT_DIR.relative_to(ROOT)}/")


if __name__ == "__main__":
    main()

Run date: 2026-10-07 12:42
Sources (licence: CC BY 4.0, Ville de Montréal):
   actes-criminels.csv  retrieved 2026-10-06  <- https://donnees.montreal.ca/dataset/5829b5b0-ea6f-476f-be94-bc2b8797769a/resource/c6f482bf-bf0f-4960-8b2f-9982c211addd/download/actes-criminels.csv
   bourough-geo-coords.json  retrieved 2026-09-24  <- https://donnees.montreal.ca/dataset/9797a946-9da8-41ec-8815-f6b276dec7e9/resource/e18bfd07-edc8-4ce8-8a5a-3b617662a794/download/limites-administratives-agglomeration.geojson



Rows in source file (all years): 361,399
Rows in 2023-2025: 97,652
   2023: 34,966
   2024: 32,854
   2025: 29,832

No usable location (missing or placeholder coordinates): 15,990 (16.4%)
   by category:
      Vol dans / sur véhicule à moteur: 5,660
      Vol de véhicule à moteur: 4,356
      Méfait: 3,457
      Introduction: 1,468
      Vols qualifiés: 1,031
      Infractions entrainant la mort: 18



Points on a borough boundary (matched 2 boroughs, kept first): 0
Located but outside the 19 boroughs: 7,212
   in a linked city (ville liée), e.g. Westmount, Mont-Royal: 7,212
      Pointe-Claire: 1,708
      Dorval: 1,575
      Dollard-des-Ormeaux: 1,050
      Westmount: 640
      Côte-Saint-Luc: 422
      Mont-Royal: 396
      Kirkland: 388
      Montréal-Est: 297
      Beaconsfield: 270
      Sainte-Anne-de-Bellevue: 136
      Hampstead: 123
      Montréal-Ouest: 112
      Baie-D'Urfé: 73
      Senneville: 22
   outside all agglomeration polygons (water, edge cases): 0

Kept records attributed to PDQ 50 (metro, placed at station entrances): 737
KEPT in the 19 boroughs: 74,450 of 97,652 (76.2%)

category                                  Break-in  Mischief  Offence causing death  Robbery  Theft from vehicle  Vehicle theft  Total
borough                                                                                                                               
Ahuntsic-Cartierville 